# 08 — Decision Simulation

The question this project actually cares about: **given only enough capacity to intervene on N patients, which N should get the intervention, and how much better is that than simpler rules?**

**Honesty check first:** this dataset has no ground-truth intervention outcomes (see `data/README.md`). Everything below is a labeled *simulation* — intervention cost, effectiveness, and readmission cost are explicit assumptions, swept over ranges, never presented as measured fact.

In [1]:
import sys
sys.path.insert(0, '../src')
import joblib
import pandas as pd
from optimization import expected_net_benefit, compare_strategies

pipe = joblib.load('../data/processed/models/xgboost.joblib')
X_train, X_test, y_train, y_test = joblib.load('../data/processed/models/splits.joblib')
risk = pipe.predict_proba(X_test)[:, 1]
y_true = y_test.values
print(f'{len(risk)} patients in the decision-simulation test set, {y_true.sum()} actual 30-day readmissions')

20075 patients in the decision-simulation test set, 2305 actual 30-day readmissions


## Assumptions

| Parameter | Value | Basis |
|---|---|---|
| Readmission cost | $10,000 | Rough order-of-magnitude, published estimates of an avoidable inpatient readmission range roughly $8K-$15K — used as a round illustrative number, not a claim about any specific hospital |
| Intervention cost | $100 | Illustrative cost of a care-management outreach/follow-up call program |
| Intervention effectiveness | 20% | Illustrative reduction in readmission probability for a patient who receives the intervention — swept in sensitivity analysis below, not fit from data |
| Capacity | 500 patients | Illustrative monthly care-management capacity |

## Example: expected net benefit for two patients

In [2]:
import numpy as np
example_risk = np.array([0.80, 0.50])
nb = expected_net_benefit(example_risk, intervention_effectiveness=0.20, readmission_cost=10_000, intervention_cost=100)
for r, n in zip(example_risk, nb):
    print(f'Risk={r:.0%} -> expected benefit=${r*0.20*10_000:,.0f}, net benefit after $100 intervention cost = ${n:,.0f}')

Risk=80% -> expected benefit=$1,600, net benefit after $100 intervention cost = $1,500
Risk=50% -> expected benefit=$1,000, net benefit after $100 intervention cost = $900


## Strategy comparison at capacity=500, uniform effectiveness

With a constant effectiveness assumption, "highest predicted risk" and "highest expected net benefit" are mathematically the same ranking — net benefit is a monotonic transform of risk when effectiveness and costs don't vary by patient. This comparison exists mainly to establish random-selection as the floor.

In [3]:
uniform_comparison = compare_strategies(y_true, risk, capacity=500)
uniform_comparison

,patients_selected,actual_readmissions_among_selected,expected_readmissions_prevented,intervention_cost,expected_savings,expected_net_benefit,roi_pct
strategy,,,,,,,
random,500,63,12.6,50000,126000.0,76000.0,152.0
highest_risk,500,196,39.2,50000,392000.0,342000.0,684.0
utility_optimized,500,196,39.2,50000,392000.0,342000.0,684.0


Even simple highest-risk targeting captures **4.5x** the net benefit of random selection at the same capacity — the value here is mostly in *having a risk model at all*, not yet in sophisticated targeting. The interesting question — does utility-based targeting beat pure risk-based targeting — needs effectiveness to actually vary by patient. That's `09_optimization.ipynb`.